In [2]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Fake News Detector: Model Training\n",
    "\n",
    "This notebook downloads the LIAR dataset, preprocesses it, trains a Logistic Regression classifier, and saves the model and vectorizer."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "import os\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "from sklearn.feature_extraction.text import TfidfVectorizer\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.metrics import classification_report, accuracy_score, f1_score\n",
    "import joblib\n",
    "import requests\n",
    "import zipfile"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Download and Extract LIAR Dataset"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "DATA_DIR = '../data'\n",
    "os.makedirs(DATA_DIR, exist_ok=True)\n",
    "url = 'https://www.cs.ucsb.edu/~william/data/liar_dataset.zip'\n",
    "zip_path = os.path.join(DATA_DIR, 'liar_dataset.zip')\n",
    "if not os.path.exists(os.path.join(DATA_DIR, 'train.tsv')):\n",
    "    print('Downloading LIAR dataset...')\n",
    "    r = requests.get(url)\n",
    "    with open(zip_path, 'wb') as f:\n",
    "        f.write(r.content)\n",
    "    with zipfile.ZipFile(zip_path, 'r') as zip_ref:\n",
    "        zip_ref.extractall(DATA_DIR)\n",
    "    print('Extracted!')\n",
    "else:\n",
    "    print('Dataset already exists.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Load Data"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "train = pd.read_csv(os.path.join(DATA_DIR, 'train.tsv'), sep='\\t', header=None)\n",
    "test = pd.read_csv(os.path.join(DATA_DIR, 'test.tsv'), sep='\\t', header=None)\n",
    "valid = pd.read_csv(os.path.join(DATA_DIR, 'valid.tsv'), sep='\\t', header=None)\n",
    "print('Train:', train.shape, 'Test:', test.shape, 'Valid:', valid.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Preprocess Data (Keep statement and label only)"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "def preprocess(df):\n",
    "    df = df[[1, 2]].copy()\n",
    "    df.columns = ['label', 'text']\n",
    "    # Map labels: true/mostly-true/half-true -> real, others -> fake\n",
    "    real = ['true', 'mostly-true', 'half-true']\n",
    "    df['label'] = df['label'].apply(lambda x: 'real' if x in real else 'fake')\n",
    "    return df\n",
    "train_df = preprocess(train)\n",
    "test_df = preprocess(test)\n",
    "valid_df = preprocess(valid)\n",
    "print(train_df.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Vectorize Text (TF-IDF)"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)\n",
    "X_train = vectorizer.fit_transform(train_df['text'])\n",
    "X_test = vectorizer.transform(test_df['text'])\n",
    "X_valid = vectorizer.transform(valid_df['text'])\n",
    "y_train = train_df['label']\n",
    "y_test = test_df['label']\n",
    "y_valid = valid_df['label']"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Train Classifier"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "clf = LogisticRegression(max_iter=1000)\n",
    "clf.fit(X_train, y_train)\n",
    "preds = clf.predict(X_test)\n",
    "print('Accuracy:', accuracy_score(y_test, preds))\n",
    "print('F1:', f1_score(y_test, preds, pos_label='real'))\n",
    "print(classification_report(y_test, preds))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Save Model and Vectorizer"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "MODEL_DIR = '../backend/model'\n",
    "os.makedirs(MODEL_DIR, exist_ok=True)\n",
    "joblib.dump(clf, os.path.join(MODEL_DIR, 'fake_news_model.pkl'))\n",
    "joblib.dump(vectorizer, os.path.join(MODEL_DIR, 'vectorizer.pkl'))\n",
    "print('Model and vectorizer saved!')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.8"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# Fake News Detector: Model Training\n',
    '\n',
    'This notebook downloads the LIAR dataset, preprocesses it, trains a Logistic Regression classifier, and saves the model and vectorizer.']},
  {'cell_type': 'code',
   'metadata': {},
   'source': ['import os\n',
    'import pandas as pd\n',
    'import numpy as np\n',
    'from sklearn.feature_extraction.text import TfidfVectorizer\n',
    'from sklearn.linear_model import LogisticRegression\n',
    'from sklearn.metrics import classification_report, accuracy_score, f1_score\n',
    'import joblib\n',
    'import requests\n',
    'import zipfile']},
  {'cell_type': 'markdown',
   'metadata': {},
   'source': ['## Download and Extract LIAR Dataset']},
  {'cell_type': 'code',
   'metadata': {},
   'source': ["DATA_DIR = '../data'\n",
    'os.makedirs(DATA_DIR, exist_ok=True)\n',
    "url = 'https://www.cs.ucsb.edu/~william/data/liar_dataset.zip'\n",
    "zip_path